# télos MDLM: Multi-Scale Training Suites
This notebook executes the training pipeline for:
- **25M 1:45 Ratio** (1.125 Billion Tokens, 8,584 total steps — Resumed from Step 1,000)

Memory GC and `mx.clear_cache()` are enforced to keep RAM usage strictly contained on Metal GPU.

In [ ]:
import os
import sys
import time
import gc
import yaml
import math
import io
from pathlib import Path
import numpy as np

# Ensure working directory is project root
project_root = Path.cwd()
while not (project_root / "mdiff").exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
import mlx.nn as nn
from mdiff.model.mlx_components import MLXTelosTransformer, load_upscaled_weights
from mdiff.training.trainer import TelosMLXTrainer
from mdiff.data.tokenizer import load_tokenizer

def run_training_step(config_path, upscaled_source=None, resume_from=None, resume_step=0):
    print("=" * 85)
    print("STARTING TRAINING RUN: " + str(config_path))
    print("=" * 85)
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    
    model = MLXTelosTransformer(**cfg["model"])
    model.set_dtype(mx.bfloat16)
    
    if upscaled_source:
        src_ckpt, src_cfg = upscaled_source
        print("  [Net2Net] Upscaling model weights from: " + str(src_ckpt))
        load_upscaled_weights(model, cfg["model"], src_ckpt, src_cfg)
    
    if resume_from:
        print(f"  [Resume] Loading weights from {resume_from}")
        model.load_weights(resume_from, strict=False)
    
    trainer = TelosMLXTrainer(model, cfg)
    trainer.train(resume_step=resume_step)
    
    del model, trainer
    gc.collect()
    mx.clear_cache()
    print("FINISHED RUN: " + str(config_path) + "\n")

In [ ]:
# PIPELINE DEFINITION: 25M 1:45 (1.125B Tokens) - Resumed from Step 1000
start_time = time.time()

print("\n>>> Resuming 25M 1:45 from Step 1000 (Remaining: 7,584 Steps, ~6.9 hrs) <<<")
run_training_step(
    "configs/masked/25m/phase_b_25m_1to45_mlx.yaml",
    resume_from="checkpoints/masked/25m/kappa_25m_1to45_mlx/checkpoint_step_1000.safetensors",
    resume_step=1000
)

total_elapsed = (time.time() - start_time) / 3600.0
print("=" * 85)
print(f"25M 1:45 RUN COMPLETED SUCCESSFULLY IN {total_elapsed:.2f} HOURS!\n")
print("=" * 85)